### GRASS setup


In [ ]:
import os, sys, shutil, subprocess, re, glob 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GRASSBIN = "/usr/local/bin/grass85"
GISDBASE = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks"
py_path  = subprocess.check_output([GRASSBIN, "--config", "python_path"], text=True).strip()
dist_dir = str(Path(py_path).parents[1])  # .../dist.x86_64-pc-linux-gnu

env = os.environ.copy()
env["GISBASE"] = dist_dir                      # ✅ IMPORTANT
env["PATH"] = f"{dist_dir}/bin:{dist_dir}/scripts:" + env["PATH"]
env["PYTHONPATH"] = py_path + os.pathsep + env.get("PYTHONPATH","")
env["LD_LIBRARY_PATH"] = f"{dist_dir}/lib:" + env.get("LD_LIBRARY_PATH","")

os.environ.update(env)
sys.path.insert(0, py_path)

import grass.script as gs
import grass.jupyter as gj



### Create New project

In [ ]:
#gs.create_project("DEM_generation_lure", epsg=3358, overwrite=True)

In [ ]:
gj.init(GISDBASE, location="DEM_generation_lure", mapset="PERMANENT", grass_path=GRASSBIN)

### Create new mapset

In [ ]:
#gs.run_command("g.mapset", flags="c", mapset="DTM_DSM_lure", overwrite=True)

In [ ]:
#gj.init(GISDBASE, location="DEM_generation", mapset="DTM_DSM", grass_path=GRASSBIN)

In [ ]:
!g.region raster=boundary

In [ ]:
gs.run_command("r.mask", raster="boundary", overwrite=True)

### Import 2017 lidar DTM (Ground+Road) Lure

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DTM_ground_water_road")


In [ ]:
folder = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DTM_ground_water_road"
tif_files = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]

imported_rasters = []
for tif in tif_files:
    full_path = os.path.join(folder, tif)
    raster_name = os.path.splitext(tif)[0]
    gs.run_command(
        "r.import",
        input=full_path,
        output=raster_name,
        resample="bilinear",   # for orthos; use "nearest" for categorical
        overwrite=True
    )
    imported_rasters.append(raster_name)

print(imported_rasters)

In [ ]:
gs.run_command("r.patch",
               input=",".join(reversed(imported_rasters)),
               output="lidar_2017_DTM_lure",
               overwrite=True)


In [ ]:
gs.run_command("r.relief", input="lidar_2017_DTM_lure", output="lidar_2017_DTM_lure_relief")

In [ ]:
gs.run_command("r.colors", map="lidar_2017_DTM_lure", color="elevation", overwrite =True )

In [ ]:
m = gj.Map(use_region=True)

m.d_rast(map="lidar_2017_DTM_lure_relief")
m.save(filename="lure_releif_2017_lidar_dtm.png")
m.show()

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="lidar_2017_DTM_lure", shade="lidar_2017_DTM_lure_relief", brighten=50)
#m.d_rast(map="lidar_2017_DTM")
m.d_legend(raster="lidar_2017_DTM_lure", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=10)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.show()
m.save("DEM_lidar_2017.png")

### Import 2017 lidar DSM Lure

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DSM_allclassified")

In [ ]:
folder = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/NOAA_lidar_2017/lidar_raster/interpolated/DSM_allclassified"
tif_files = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]

imported_DSM_rasters = []
for tif in tif_files:
    full_path = os.path.join(folder, tif)
    raster_name = os.path.splitext(tif)[0]
    gs.run_command(
        "r.import",
        input=full_path,
        output=raster_name,
        resample="bilinear",   # for orthos; use "nearest" for categorical
        overwrite=True
    )
    imported_DSM_rasters.append(raster_name)

print(imported_DSM_rasters)


In [ ]:
gs.run_command("r.patch",
               input=",".join(reversed(imported_DSM_rasters)),
               output="lidar_2017_DSM_lure",
               overwrite=True)


In [ ]:
gs.run_command("r.relief", input="lidar_2017_DSM_lure", output="lidar_2017_DSM_lure_relief")

In [ ]:
m = gj.Map()
m.d_shade(color="lidar_2017_DSM_lure", shade="lidar_2017_DSM_lure_relief", brighten=50)
#m.d_rast(map="lidar_2017_DTM")
m.d_legend(raster="lidar_2017_DSM_lure", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
m.d_barscale(at=(1,11), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.show()

### Data Import SFM 2024(Just after helene)

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Aerial_photogrammetry")

In [ ]:
#!g.region -p

##### Import and create DSM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="A0069A_data_upload_chimney_sfm_med_point_cloud.copc.laz",
    output="sfm_pc_DSM_lure",
    #class_filter="2,11",  # Ground and Roads
    overwrite=True
)

In [ ]:
gs.run_command(
    "v.surf.rst",
    input="sfm_pc_DSM_lure",
    elevation="sfm_dsm_lure",
    smooth=1,
    dmin=1,
    segmax=40,
    tension=20,
    npmin=120, 
    overwrite=True
)

In [ ]:
    gs.run_command(
        "r.fillnulls",
        input="sfm_dsm_lure",
        output="sfm_dsm_filled_lure",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="sfm_dsm_filled_lure", output="sfm_DSM_relief_filled_lure")

In [ ]:
gs.run_command("r.colors", map="sfm_dsm_filled_lure", color="elevation", overwrite =True )

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="sfm_dsm_filled_lure", shade="sfm_DSM_relief_filled_lure", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
#m.d_rast(map="sfm_DSM_relief")
m.d_legend(raster="sfm_dsm_filled_lure", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=10)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.show()
m.save("DEM_sfm_2024.png")

##### Import ground and road for DTM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="A0069A_data_upload_chimney_sfm_med_point_cloud.copc.laz",
    output="sfm_DTM_lure",
    class_filter="2,11",  # Ground and Roads
    overwrite=True
)

In [ ]:
## used corey's parameters
gs.run_command(
    "v.surf.rst",
    input="sfm_DTM_lure",
    elevation="sfm_dtm_lure",
    smooth=1,
    dmin=1,
    segmax=40,
    tension=20,
    npmin=120,
    overwrite=True
)

In [ ]:
## parameters suggested by helena
gs.run_command(
    "v.surf.rst",
    input="sfm_DTM_lure",
    elevation="sfm_dtm_lure",
    smooth=1,
    dmin=2,
    segmax=40,
    tension=20,
    npmin=300,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="sfm_dtm_lure",
        output="sfm_dtm_filled_lure",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="sfm_dtm_filled_lure", output="sfm_dtm_relief_filled_lure")

In [ ]:
m = gj.Map(use_region=True)
#m.d_shade(color="sfm_dtm", shade="sfm_dtm_relief", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
m.d_rast(map="sfm_dtm_relief_filled_lure")
#m.d_legend(raster="sfm_dtm", at="12,45,2,9", flags="b", font="Fira Sans Condensed Light", fontsize=7)
#m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=7)
m.save(filename="lure_releif_2024_sfm_dtm.png")
m.show()

In [ ]:
gs.run_command("r.colors", map="sfm_dtm_filled_lure", color="elevation", overwrite =True )

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="sfm_dtm_filled_lure", shade="sfm_dtm_relief_filled_lure", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
#m.d_rast(map="sfm_DSM_relief")
m.d_legend(raster="sfm_dtm_filled_lure", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=10)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.show()
m.save("DEM_sfm_2024.png")

### Data Import lidar 2024 (After helene from will)

In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks/written_exam/grass_project_correction/data")

##### Import dtm data and create DTM

In [ ]:
!g.region res=1

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="after_lidar_grass.las",
    output="lidar_2024_dtm_allpoints_lure",
    class_filter="2,11",
    flags="wr",  
    overwrite=True
)

In [ ]:
gs.run_command(
    "v.surf.rst",
    input="lidar_2024_dtm_allpoints_lure",
    elevation="lidar_2024_dtm_allpoints_lure",
    smooth=1,
    dmin=2,
    segmax=40,
    tension=20,
    npmin=300,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="lidar_2024_dtm_allpoints_lure",
        output="lidar_2024_dtm_allpoints_filled_lure",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="lidar_2024_dtm_allpoints_filled_lure", output="lidar_2024_dtm_relief_allpoints_filled_lure")

In [ ]:
m = gj.Map()
m.d_rast(map="lidar_2024_dtm_relief_allpoints_filled_lure")
m.show()
m.save(filename="lure_releif_2024_lidar_dtm.png")

In [ ]:
gs.run_command("r.colors", map="lidar_2024_dtm_allpoints_filled_lure", color="elevation", overwrite =True )

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="lidar_2024_dtm_allpoints_filled_lure", shade="lidar_2024_dtm_relief_allpoints_filled_lure", brighten=50)
#m.d_rast(map="lidar_2020_DTM")
#m.d_rast(map="sfm_DSM_relief")
m.d_legend(raster="lidar_2024_dtm_allpoints_filled_lure", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=10)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.show()
m.save("DEM_lidar_2024.png")

##### Import dsm data and create DSM

In [ ]:
gs.run_command(
    'v.in.pdal',
    input="after_lidar_grass.las",
    output="lidar_2024_dsm_lure",
    flags="wr",
    overwrite=True
)

In [ ]:
gs.run_command(
    "v.surf.rst",
    input="lidar_2024_dsm_lure",
    elevation="lidar_2024_dsm_lure",
    smooth=1,
    dmin=2,
    segmax=40,
    tension=20,
    npmin=300,
    overwrite=True
)

In [ ]:
gs.run_command(
        "r.fillnulls",
        input="lidar_2024_dsm_lure",
        output="lidar_2024_dsm_filled_lure",
        method="rst",
        tension=10,
        smooth=1,
        edge=15,
        npmin=100,
        segmax=80,
        memory=5000,
        overwrite=True,
    )

In [ ]:
gs.run_command("r.relief", input="lidar_2024_dsm_filled_lure", output="lidar_2024_dsm_relief_filled_lure")

In [ ]:
m = gj.Map(use_region=True)
m.d_rast(map="lidar_2024_dsm_relief_filled_lure")
m.show()

### Create line for reference points for vertical correction


In [ ]:
# Set working directory to the folder where the data is
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Orthos")

In [ ]:
gs.run_command(
        "r.import",           
        input= "Ortho_aerial_2024.tif",
        output="Ortho_aerial_2024",
        resample="bilinear",   # good default for orthophotos
        overwrite=True
    )

In [ ]:
gs.run_command(
    "r.composite",
    red="Ortho_aerial_2024.1",
    green="Ortho_aerial_2024.2",
    blue="Ortho_aerial_2024.3",
    output="Ortho_aerial_2024",
    overwrite=True
)

In [ ]:
#!g.list type=raster

In [ ]:
map = gj.InteractiveMap()
#map.add_raster("Ortho_aerial_2024_rgb")
map.add_raster("Ortho_aerial_2024@PERMANENT")
map.add_vector("reg_points")
map.show()

### Generate profile before vertical correction



In [ ]:
os.chdir("/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks")

In [ ]:
gj.init(GISDBASE, location="DEM_generation_lure", mapset="PERMANENT", grass_path=GRASSBIN)

In [ ]:
!g.list type=raster

In [ ]:
points_vec = "reg_points"  
rasters = [
    "lidar_2017_DTM_lure","lidar_2017_DSM_lure",
    "lidar_2024_dtm_allpoints_filled_lure","lidar_2024_dsm_filled_lure",
    "sfm_dtm_filled_lure","sfm_dsm_filled_lure",
]

# --- Export points as ASCII (x y cat) and parse coordinates ---
# v.out.ascii default format is point: x|y|cat (separator can vary); use a robust regex
txt = gs.read_command("v.out.ascii", input=points_vec).strip().splitlines()
if not txt:
    raise RuntimeError(f"No points found in {points_vec}")

pts = []
for line in txt:
    nums = re.findall(r"-?\d+(?:\.\d+)?", line)
    if len(nums) < 2:
        continue
    x, y = float(nums[0]), float(nums[1])
    cat = int(float(nums[2])) if len(nums) >= 3 else None
    pts.append((cat, x, y))

if len(pts) < 2:
    raise RuntimeError(f"Need at least 2 points for r.profile, found {len(pts)} in {points_vec}")

# Sort points by category when available; otherwise keep as-is
if all(p[0] is not None for p in pts):
    pts.sort(key=lambda t: t[0])

# Remove consecutive duplicates (optional)
coords_list = []
prev = None
for _, x, y in pts:
    cur = (x, y)
    if prev != cur:
        coords_list.append(f"{x},{y}")
        prev = cur

coords = ",".join(coords_list)

print("Points used:", len(coords_list))
print("First:", coords_list[0], " Last:", coords_list[-1])

out_dir = Path(f"./profiles_out_lure_{points_vec}")
out_dir.mkdir(parents=True, exist_ok=True)

for r in rasters:
    out_csv = out_dir / f"profile_{r}.csv"
    gs.run_command(
        "r.profile",
        input=r,
        coordinates=coords,
        resolution=1,
        output=str(out_csv),
        flags="g",
        overwrite=True,
    )
    print("Wrote:", out_csv)

print("Done ->", out_dir)


In [ ]:
v = "reg_points"
files = sorted(glob.glob(f"./profiles_out_lure_{v}/profile_*.csv"))
if not files: raise FileNotFoundError(f"No CSVs in ./profiles_out_{v}/")

def load(f):
    df = pd.read_csv(f, header=None, comment="#", sep=None, engine="python")
    if df.shape[1] == 1: df = df[0].astype(str).str.strip().str.split(r"[,\s]+", expand=True)
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    if df.shape[1] >= 3:
        x,y,z = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,-1].to_numpy()
        d = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        return d, z
    return df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy()

plt.figure(figsize=(18,6))
for f in files:
    d,z = load(f)
    plt.plot(d, z, label=re.sub(r"^profile_|\.csv$", "", os.path.basename(f)))
plt.grid(True, alpha=.4); plt.xlabel("Distance (m)"); plt.ylabel("Elevation")
plt.legend(loc="upper left", bbox_to_anchor=(1.02,1.0))
plt.savefig(f"lure_profile_plot_{v}.png", dpi=300, bbox_inches="tight"); plt.show()

### We can see clearly that there is some vertical shift in lidar 2024 DSM and DTM, Now we need to correct that.
We will take 2017 lidar data as reference and correct all according to that. We will use stable road points for the correction.

In [ ]:
!g.list type=raster

In [ ]:
from pandas import json_normalize
PTS   = "reg_points"
R_REF = "lidar_2017_DTM_lure"   # <-- reference (no 2020)

TARGETS = [
    "lidar_2024_dtm_allpoints_filled_lure",
    "lidar_2024_dsm_filled_lure",
    "sfm_dtm_filled_lure",
    "sfm_dsm_filled_lure"
]

SUFFIX = "_vref2017_rmdse"

def signed_rmdse_bias(dem1, dem2, control_points):
    data = gs.parse_command("r.what", flags="n", map=f"{dem2},{dem1}",
                            points=control_points, format="json")
    df = json_normalize(data)
    a, b = f"{dem1}.value", f"{dem2}.value"
    df[a] = np.array(df[a], float); df[b] = np.array(df[b], float)
    df = df[np.isfinite(df[a]) & np.isfinite(df[b])].copy()
    diff = df[b] - df[a]                # target - reference
    med  = float(np.median(diff))
    mag  = float(np.sqrt(np.median(diff**2)))   # RMdSE
    signed_bias = (1.0 if med >= 0 else -1.0) * mag
    return signed_bias, mag, med

def outname(r, s): return f"{r.split('@')[0]}{s}"

gs.run_command("g.region", raster=R_REF)

for tgt in TARGETS:
    out = outname(tgt, SUFFIX)
    signed_bias, mag, med = signed_rmdse_bias(R_REF, tgt, PTS)
    gs.mapcalc(f"{out} = {tgt} - ({signed_bias})", overwrite=True)
    new_signed_bias, new_mag, new_med = signed_rmdse_bias(R_REF, out, PTS)

    print(f"\n--- {tgt} -> {out} ---")
    print(f"median(diff)={med:.4f}  RMdSE={mag:.4f}  signed_bias={signed_bias:.4f}")
    print(f"after: median(diff)={new_med:.4f}  RMdSE={new_mag:.4f}  signed_bias={new_signed_bias:.4f}")

print("\nDONE.")


#### Now new profile after correction

In [ ]:
#g.list type=vector

In [ ]:
rename_pairs = {
    "lidar_2017_DTM":                               "lidar_2017_DTM_ref",
    "lidar_2024_dtm_allpoints_filled_lure_vref2017_rmdse": "lidar_2024_DTM_corr_lure",
    "lidar_2024_dsm_filled_lure_vref2017_rmdse":           "lidar_2024_DSM_corr_lure",
    "sfm_dtm_filled_lure_vref2017_rmdse":                  "sfm_DTM_corr_lure",
    "sfm_dsm_filled_lure_vref2017_rmdse":                  "sfm_DSM_corr_lure",
}

for old, new in rename_pairs.items():
    print(f"Renaming {old} -> {new}")
    gs.run_command("g.rename", raster=f"{old},{new}", overwrite=True)

rasters = [
    "lidar_2017_DTM_ref",
    "lidar_2024_DTM_corr_lure",
    "lidar_2024_DSM_corr_lure",
    "sfm_DTM_corr_lure",
    "sfm_DSM_corr_lure",
]
print(rasters)


In [ ]:
points_vec = "reg_points"

rasters = [
    "lidar_2017_DTM_lure",
    "lidar_2024_DTM_corr_lure",
    "lidar_2024_DSM_corr_lure",
    "sfm_DTM_corr_lure",
    "sfm_DSM_corr_lure",
]

txt = gs.read_command("v.out.ascii", input=points_vec).strip().splitlines()
if not txt: raise RuntimeError(f"No points found in {points_vec}")

pts = []
for ln in txt:
    nums = re.findall(r"-?\d+(?:\.\d+)?", ln)
    if len(nums) >= 2:
        x, y = float(nums[0]), float(nums[1])
        cat = int(float(nums[2])) if len(nums) >= 3 else None
        pts.append((cat, x, y))

if len(pts) < 2: raise RuntimeError(f"Need >=2 points in {points_vec}, found {len(pts)}")
if all(p[0] is not None for p in pts): pts.sort(key=lambda t: t[0])

coords_list, prev = [], None
for _, x, y in pts:
    if prev != (x, y):
        coords_list.append(f"{x},{y}")
        prev = (x, y)
coords = ",".join(coords_list)

out_dir = Path(f"./profiles_out_vref2017_lure_{points_vec}")
out_dir.mkdir(parents=True, exist_ok=True)

for r in rasters:
    out_csv = out_dir / f"profile_{r}.csv"
    gs.run_command("r.profile", input=r, coordinates=coords, resolution=1,
                   output=str(out_csv), flags="g", overwrite=True)
    print("Wrote:", out_csv)

print("Done ->", out_dir)


In [ ]:
points_vec = "reg_points"
files = sorted(glob.glob(f"./profiles_out_vref2017_lure_{points_vec}/profile_*.csv"))
if not files: raise FileNotFoundError("No profile_*.csv found")

def load_profile(f):
    df = pd.read_csv(f, header=None, comment="#", sep=None, engine="python")
    if df.shape[1] == 1:
        s = df[0].astype(str).str.strip()
        s = s[(s != "") & (~s.str.startswith("#"))]
        df = s.str.split(r"[,\s]+", expand=True)
    df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")
    if df.shape[1] >= 3:
        x,y,v = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,-1].to_numpy()
        d = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        return d, v
    return df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy()

plt.figure(figsize=(18,6))
for f in files:
    d,v = load_profile(f)
    plt.plot(d, v, label=re.sub(r"^profile_|\.csv$", "", os.path.basename(f)))

plt.xlabel("Distance (m)"); plt.ylabel("Elevation")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper left", bbox_to_anchor=(1.02,1.0), frameon=True)
plt.savefig(f"profile_plot_after_vertical_correction_vref2017_lure_{points_vec}.png",
            dpi=300, bbox_inches="tight")
plt.show()

### Since the data are properly coregistered, now they are ready for analysis

In [ ]:
#gs.run_command("g.mapset", flags="c", mapset="Analysis_lure", overwrite=True)

In [ ]:
gj.init(GISDBASE, location="DEM_generation_lure", mapset="Analysis_lure", grass_path=GRASSBIN)

In [ ]:
!g.region raster=boundary

In [ ]:
gs.run_command("r.mask", raster="boundary", overwrite=True)

### Compute DOD lidar 2017 and lidar 2024

In [ ]:
gs.mapcalc("Change_in_DTM_lidar_lure = lidar_2024_DTM_corr_lure-lidar_2017_DTM_lure", overwrite=True)

In [ ]:
gs.run_command("r.relief", input="Change_in_DTM_lidar_lure", output="Change_in_DTM_lidar_relief_lure",overwrite=True)

In [ ]:
gs.run_command(
    "r.colors",
    map="Change_in_DTM_lidar_lure",
    color="differences",
)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="Change_in_DTM_lidar_lure", shade="Change_in_DTM_lidar_relief_lure", brighten=40)
#m.d_vect(map="basins_90_v_cat34", color="green", width=2, fill_color="none")
m.d_legend(raster="Change_in_DTM_lidar_lure", flags="-d", title_fontsize=10, at="6,11,97,60")
m.d_barscale(at=(1,12), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("change_dtm_lidar_lure.jpg")
m.show()


### Compute DOD lidar 2017 and sfm 2024

In [ ]:
gs.mapcalc("Change_in_DTM_lidar_sfm_lure = sfm_DTM_corr_lure-lidar_2017_DTM_lure", overwrite=True)

In [ ]:
gs.run_command("r.relief", input="Change_in_DTM_lidar_sfm_lure", output="Change_in_DTM_lidar_sfm_relief_lure",overwrite=True)

In [ ]:
gs.run_command(
    "r.colors",
    map="Change_in_DTM_lidar_sfm_lure",
    color="differences",
)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="Change_in_DTM_lidar_sfm_lure", shade="Change_in_DTM_lidar_sfm_relief_lure", brighten=40)

#m.d_vect(map="basins_90_v_cat34", color="green", width=2, fill_color="none")
#m.d_legend(raster="Change_in_DTM_lidar_sfm_lure", title="Elevation Change",font="Fira Sans Condensed Light",fontsize=14, flags="-d", title_fontsize=12)
m.d_legend(raster="Change_in_DTM_lidar_sfm_lure", flags="-d", title_fontsize=10, at="6,11,97,60")
m.d_barscale(at=(1,12), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("change_dtm_sfm.jpg")
m.show()

### Overland Flow Maps

#### 2017 lidar

In [ ]:
!g.region raster=morse_park_rast  # Morse park area only

In [ ]:
gs.run_command("r.mask", raster="morse_park_rast", overwrite=True)

In [ ]:
!g.region res=1

In [ ]:
gs.run_command(
    "r.slope.aspect",
    elevation="lidar_2017_DTM_lure",
    dx="dx_2017_lure",
    dy="dy_2017_lure",
)

In [ ]:
#gs.run_command('r.sim.water', elevation="lidar_2017_DTM_lure", dx="dx_2017_lure", dy="dy_2017_lure", depth="depth_map_2017_lure", overwrite=True)

In [ ]:
reg = gs.parse_command("g.region", flags="g")
cells = int(reg["rows"]) * int(reg["cols"])

# Rule of thumb for 1 m AOI: at least 200k, up to ~1.2M
nwalkers = min(1_200_000, max(200_000, cells // 2))
print("cells:", cells, "nwalkers:", nwalkers)

In [ ]:
# 3) Run SIMWE and SAVE DISCHARGE (this is the key drainage product)
gs.run_command(
    "r.sim.water",
    elevation="lidar_2017_DTM_lure",
    dx="dx_2017_lure",
    dy="dy_2017_lure",
    depth="depth_2017_lidar_morse",
    discharge="q_2017_lidar_morse",
    rain_value=60,
    infil_value=0,
    man_value=0.05,
    niterations=45,
    # output_step=1,
    # mintimestep=10,
    # diffusion_coeff=0.8,
    # hmax=0.2,
    overwrite=True,
)

In [ ]:
#!g.list type=raster

In [ ]:
gs.run_command("r.relief", input="depth_2017_lidar_morse", output="depth_map_2017_lure_relief",overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="depth_2017_lidar_morse", shade="depth_map_2017_lure_relief", brighten=80)
#m.d_rast(map="q_2017_lidar_morse")
m.d_legend(raster="depth_2017_lidar_morse", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=8, label_step=1)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("depth_dtm_2017_lidar_morse.jpg")
m.show()

#### 2024 lidar

In [ ]:
!g.region res=1

In [ ]:

gs.run_command(
    "r.slope.aspect",
    elevation="lidar_2024_DTM_corr_lure",
    dx="dx_2024_lidar_lure",
    dy="dy_2024_lidar_lure",
)

In [ ]:
#gs.run_command('r.sim.water', elevation="lidar_2024_DTM_corr_lure", dx="dx_2024_lidar_lure", dy="dy_2024_lidar_lure", depth="depth_map_2024_lidar_lure", niterations=30, overwrite=True, rain_value=200, infil_value=0, man_value=0.05, nwalkers=10000)
gs.run_command('r.sim.water', elevation="lidar_2024_DTM_corr_lure", dx="dx_2024_lidar_lure", dy="dy_2024_lidar_lure", depth="depth_map_2024_lidar_lure",rain_value=60,infil_value=0,man_value=0.05,niterations=45,overwrite=True)

In [ ]:
gs.run_command("r.relief", input="depth_map_2024_lidar_lure", output="depth_map_2024_lidar_lure_relief",overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="depth_map_2024_lidar_lure", shade="depth_map_2024_lidar_lure_relief", brighten=80)
#m.d_rast(map="depth_map")
m.d_legend(raster="depth_map_2024_lidar_lure", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=8,label_step=1)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("depth_dtm_2024_lidar_morse.jpg")
m.show()

#### 2024 sfm

In [ ]:

gs.run_command(
    "r.slope.aspect",
    elevation="sfm_DTM_corr_lure",
    dx="dx_2024_sfm_lure",
    dy="dy_2024_sfm_lure",
)

In [ ]:
#gs.run_command('r.sim.water', elevation="sfm_DTM_corr_lure", dx="dx_2024_sfm_lure", dy="dy_2024_sfm_lure", depth="depth_map_2024_sfm_lure", niterations=30, overwrite=True, rain_value=200, infil_value=0, man_value=0.05, nwalkers=10000)
gs.run_command('r.sim.water', elevation="sfm_DTM_corr_lure", dx="dx_2024_sfm_lure", dy="dy_2024_sfm_lure", depth="depth_map_2024_sfm_lure", discharge="discharge_map_2024_sfm_lure",rain_value=60,  infil_value=0, man_value=0.05, niterations=45,overwrite=True)

In [ ]:
gs.run_command("r.relief", input="depth_map_2024_sfm_lure", output="depth_map_2024_sfm_lure_relief",overwrite=True)
#gs.run_command("r.relief", input="discharge_map_2024_sfm_lure", output="discharge_map_2024_sfm_lure_relief",overwrite=True)

In [ ]:
m = gj.Map(use_region=True)
m.d_shade(color="depth_map_2024_sfm_lure", shade="depth_map_2024_sfm_lure_relief", brighten=80)
#m.d_rast(map="depth_map")
m.d_legend(raster="depth_map_2024_sfm_lure", at="13,65,3,5", flags="b", font="Fira Sans Condensed Light", fontsize=8, label_step=1)
m.d_barscale(at=(1,9), flags="n", font="Fira Sans Condensed Light", fontsize=10)
m.save("depth_dtm_2024_sfm_lure.jpg")
m.show()

### Overland flow in morse park

In [ ]:
#g.region

### Volume Computation